In [2]:
 #Imports and configuration
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
from config import RAW, PROCESSED, SEED

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("Données brutes :", RAW)
#Import libraries and load the shared config (paths and random seed)

Données brutes : C:\Users\USER\anaconda_projects\ae0031cb-ad1c-4e8e-b428-a8bce55a9655\data\raw


In [3]:
#Load the 5 CSV files
tel  = pd.read_csv(RAW / "PdM_telemetry.csv", parse_dates=["datetime"])
err  = pd.read_csv(RAW / "PdM_errors.csv",    parse_dates=["datetime"])
mnt  = pd.read_csv(RAW / "PdM_maint.csv",     parse_dates=["datetime"])
fail = pd.read_csv(RAW / "PdM_failures.csv",  parse_dates=["datetime"])
mach = pd.read_csv(RAW / "PdM_machines.csv")

tables = {"telemetry": tel, "errors": err, "maint": mnt, "failures": fail, "machines": mach}
for name, df in tables.items():
    print(f"{name:10s} {df.shape}")
#Load telemetry, errors, maintenance, failures, and machines with datetime parsing

telemetry  (876100, 6)
errors     (3919, 3)
maint      (3286, 3)
failures   (761, 3)
machines   (100, 3)


In [4]:
#Structure and types
for name, df in tables.items():
    print("=" * 60)
    print(name.upper(), df.shape)
    print(df.dtypes.to_string())
    display(df.head(3))
#Inspect columns, data types, and preview the first rows of each table

TELEMETRY (876100, 6)
datetime     datetime64[us]
machineID             int64
volt                float64
rotate              float64
pressure            float64
vibration           float64


,datetime,machineID,volt,rotate,pressure,vibration
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847


ERRORS (3919, 3)
datetime     datetime64[us]
machineID             int64
errorID                 str


,datetime,machineID,errorID
0,2015-01-03 07:00:00,1,error1
1,2015-01-03 20:00:00,1,error3
2,2015-01-04 06:00:00,1,error5


MAINT (3286, 3)
datetime     datetime64[us]
machineID             int64
comp                    str


,datetime,machineID,comp
0,2014-06-01 06:00:00,1,comp2
1,2014-07-16 06:00:00,1,comp4
2,2014-07-31 06:00:00,1,comp3


FAILURES (761, 3)
datetime     datetime64[us]
machineID             int64
failure                 str


,datetime,machineID,failure
0,2015-01-05 06:00:00,1,comp4
1,2015-03-06 06:00:00,1,comp1
2,2015-04-20 06:00:00,1,comp2


MACHINES (100, 3)
machineID    int64
model          str
age          int64


,machineID,model,age
0,1,model3,18
1,2,model4,7
2,3,model3,8


In [5]:
# Missing values and duplicates
audit = []
for name, df in tables.items():
    audit.append({"table": name, "lignes": len(df),
                  "NaN total": int(df.isna().sum().sum()),
                  "lignes dupliquées": int(df.duplicated().sum())})
audit = pd.DataFrame(audit)
display(audit)

# doublons sur la clé (datetime, machineID) dans la télémétrie
print("Doublons de clé (datetime, machineID) dans telemetry :",
      tel.duplicated(subset=["datetime", "machineID"]).sum())
#Check for missing values and duplicate rows in all tables; also check duplicate keys in telemetry.

,table,lignes,NaN total,lignes dupliquées
0,telemetry,876100,0,0
1,errors,3919,0,0
2,maint,3286,0,0
3,failures,761,0,0
4,machines,100,0,0


Doublons de clé (datetime, machineID) dans telemetry : 0


In [6]:
#Time coverage and regularity
for name in ["telemetry", "errors", "maint", "failures"]:
    df = tables[name]
    print(f"{name:10s} {df['datetime'].min()}  ->  {df['datetime'].max()}")

per_machine = tel.groupby("machineID").size()
print("\nMesures par machine : min =", per_machine.min(), "| max =", per_machine.max())
print("Nombre de machines :", tel["machineID"].nunique())

steps = (tel.sort_values(["machineID", "datetime"])
            .groupby("machineID")["datetime"].diff().dropna())
print("\nPas entre mesures consécutives :")
print(steps.value_counts().head())
#Verify the time range, number of measurements per machine, and the hourly frequency

telemetry  2015-01-01 06:00:00  ->  2016-01-01 06:00:00
errors     2015-01-01 06:00:00  ->  2016-01-01 05:00:00
maint      2014-06-01 06:00:00  ->  2016-01-01 06:00:00
failures   2015-01-02 03:00:00  ->  2015-12-31 06:00:00

Mesures par machine : min = 8761 | max = 8761
Nombre de machines : 100

Pas entre mesures consécutives :
datetime
0 days 01:00:00    876000
Name: count, dtype: int64


In [7]:
#Telemetry statistics
display(tel[["volt", "rotate", "pressure", "vibration"]].describe().round(2))
#Display descriptive statistics (mean, std, min, max, quartiles) for volt, rotate, pressure, vibration.

,volt,rotate,pressure,vibration
count,876100.00,876100.00,876100.00,876100.00
mean,170.78,446.61,100.86,40.39
std,15.51,52.67,11.05,5.37
min,97.33,138.43,51.24,14.88
25%,160.30,412.31,93.50,36.78
50%,170.61,447.56,100.43,40.24
75%,181.00,482.18,107.56,43.78
max,255.12,695.02,185.95,76.79


In [8]:
#Event and machine distribution
print("ERREURS par type :");   print(err["errorID"].value_counts().sort_index().to_string())
print("\nPANNES par composant :"); print(fail["failure"].value_counts().sort_index().to_string())
print("\nMAINTENANCES par composant :"); print(mnt["comp"].value_counts().sort_index().to_string())
print("\nMACHINES par modèle :");  print(mach["model"].value_counts().sort_index().to_string())
print("\nÂge des machines :");     print(mach["age"].describe().round(1).to_string())

#Show counts of errors by type, failures by component, maintenance by component, machine models, and age statistics.

ERREURS par type :
errorID
error1    1010
error2     988
error3     838
error4     727
error5     356

PANNES par composant :
failure
comp1    192
comp2    259
comp3    131
comp4    179

MAINTENANCES par composant :
comp
comp1    804
comp2    863
comp3    808
comp4    811

MACHINES par modèle :
model
model1    16
model2    17
model3    35
model4    32

Âge des machines :
count    100.0
mean      11.3
std        5.9
min        0.0
25%        6.8
50%       12.0
75%       16.0
max       20.0


In [9]:
#Referential integrity
valid_ids = set(mach["machineID"])
for name in ["telemetry", "errors", "maint", "failures"]:
    bad = ~tables[name]["machineID"].isin(valid_ids)
    print(f"{name:10s} machineID inconnus : {bad.sum()}")

keys = tel[["datetime", "machineID"]]
for name in ["errors", "maint", "failures"]:
    m = tables[name].merge(keys, on=["datetime", "machineID"], how="left", indicator=True)
    print(f"{name:10s} événements sans mesure télémétrie à la même heure : "
          f"{(m['_merge'] == 'left_only').sum()}")
#Check that all machineIDs are valid and that every event has a corresponding telemetry timestamp

telemetry  machineID inconnus : 0
errors     machineID inconnus : 0
maint      machineID inconnus : 0
failures   machineID inconnus : 0
errors     événements sans mesure télémétrie à la même heure : 0
maint      événements sans mesure télémétrie à la même heure : 400
failures   événements sans mesure télémétrie à la même heure : 0


In [10]:
#Failure ↔ maintenance link
f = fail.rename(columns={"failure": "comp"})
link = f.merge(mnt.assign(has_maint=1), on=["datetime", "machineID", "comp"], how="left")
print("Pannes avec maintenance du même composant au même instant : "
      f"{link['has_maint'].notna().mean():.1%}")

print("\nÉvénements par heure de la journée :")
for name in ["errors", "maint", "failures"]:
    print(name, tables[name]["datetime"].dt.hour.value_counts().sort_index().to_dict())

#Analyze how often a failure triggers maintenance on the same component at the same time; also show event hour distribution

Pannes avec maintenance du même composant au même instant : 97.6%

Événements par heure de la journée :
errors {0: 126, 1: 132, 2: 142, 3: 112, 4: 127, 5: 117, 6: 1122, 7: 119, 8: 135, 9: 113, 10: 124, 11: 98, 12: 120, 13: 130, 14: 128, 15: 129, 16: 113, 17: 124, 18: 127, 19: 108, 20: 127, 21: 108, 22: 116, 23: 122}
maint {6: 3286}
failures {3: 18, 6: 743}


In [11]:
#Save loaded telemetry
PROCESSED.mkdir(parents=True, exist_ok=True)
try:
    tel.to_parquet(PROCESSED / "telemetry.parquet", index=False)
    saved = "parquet"
except Exception as e:
    tel.to_pickle(PROCESSED / "telemetry.pkl")
    saved = f"pickle (parquet indisponible : {type(e).__name__})"
print("Télémétrie sauvegardée en", saved)
#Save the typed telemetry to data/processed/ as parquet (or pickle if pyarrow is not available)

Télémétrie sauvegardée en parquet
